# ? Flujos de trabajo secuenciales con modelos de GitHub (Python)

## ?? Tutorial avanzado de procesamiento secuencial

Este cuaderno demuestra **patrones de flujo de trabajo secuenciales** utilizando el Marco de Agentes de Microsoft. Aprender?s a construir sofisticadas tuber?as de procesamiento de m?ltiples pasos donde los agentes se ejecutan en un orden espec?fico, pasando datos y contexto entre etapas.

## ?? Objetivos de aprendizaje

### ?? **Patrones de procesamiento secuencial**
- **Dise?o de flujo de trabajo lineal**: Crear tuber?as de procesamiento paso a paso
- **Gesti?n del flujo de datos**: Transferir informaci?n entre agentes secuenciales
- **Procesamiento con puntos de control**: Implementar etapas de validaci?n y verificaci?n
- **Seguimiento del progreso**: Monitorear la ejecuci?n del flujo de trabajo y los resultados intermedios

### ??? **Arquitectura de tuber?as empresariales**
- **Modelado de procesos empresariales**: Mapear procesos empresariales reales a flujos de trabajo de agentes
- **Aseguramiento de calidad**: Procesos de validaci?n y revisi?n en m?ltiples etapas
- **Procesamiento de documentos**: An?lisis y transformaci?n secuencial de documentos
- **Producci?n de contenido**: Flujos editoriales con etapas de revisi?n y aprobaci?n

### ?? **Caracter?sticas avanzadas de flujo de trabajo**
- **Preservaci?n del contexto**: Mantener el estado a lo largo de las etapas del flujo de trabajo
- **Propagaci?n de errores**: Manejar fallos en el procesamiento secuencial
- **Optimizaci?n del rendimiento**: Patrones eficientes de ejecuci?n secuencial
- **Rastreo de auditor?a**: Seguimiento completo de las operaciones secuenciales

## ?? Prerrequisitos y configuraci?n

### ?? **Dependencias**
```bash
pip install agent-framework-core -U
```

### ?? **Configuraci?n**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

## ?? **Casos de uso de flujos de trabajo secuenciales empresariales**

### ?? **Tuber?a de procesamiento de documentos**
```
Raw Document ? Content Extraction ? Analysis ? Validation ? Final Output
```

### ?? **Flujo de trabajo de aseguramiento de calidad** 
```
Initial Review ? Technical Validation ? Compliance Check ? Final Approval
```

### ?? **Tuber?a de producci?n de contenido**
```
Research ? Writing ? Editing ? Review ? Publishing
```

### ?? **Automatizaci?n de procesos empresariales**
```
Data Collection ? Processing ? Analysis ? Report Generation ? Distribution
```

## ?? **Principios de dise?o de flujos de trabajo secuenciales**

- **?? Progresi?n lineal**: Cada etapa depende de la salida de la etapa anterior
- **?? Gesti?n del estado**: Preservar el contexto y los datos en todas las etapas
- **??? Manejo de errores**: Gesti?n de fallos de manera adecuada en cualquier etapa
- **?? Monitoreo del progreso**: Rastrear la finalizaci?n y el rendimiento en cada etapa
- **?? Reutilizaci?n de etapas**: Dise?ar componentes de flujo de trabajo reutilizables

?Construyamos flujos de procesamiento secuenciales sofisticados! ??


In [ ]:
! pip install agent-framework-core -U

In [ ]:
from agent_framework.openai import OpenAIChatClient
from agent_framework import ChatMessage, DataContent, Role, TextContent,WorkflowBuilder,  WorkflowEvent,WorkflowViz

In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
chat_client = OpenAIChatClient(base_url=os.environ.get("GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o" )

In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent   = chat_client.create_agent(
        instructions=(
           SalesAgentInstructions
        ),
        name=SalesAgentName,
    )

price_agent = chat_client.create_agent(
        instructions=(
            PriceAgentInstructions
        ),
        name=PriceAgentName,
    )

quote_agent = chat_client.create_agent(
        instructions=(
            QuoteAgentInstructions
        ),
        name=QuoteAgentName,
    )


In [ ]:
workflow = WorkflowBuilder().set_start_executor(sales_agent).add_edge(sales_agent, price_agent).add_edge(price_agent, quote_agent).build()

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
svg_file = viz.export(format="svg")
print(f"SVG file saved to: {svg_file}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"?? Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"? Fallback HTML render also failed: {inner}")
else:
    print("? SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
message = ChatMessage(
        role=Role.USER,
        contents=[
            TextContent(text="Please find the relevant furniture according to the image and give the corresponding price for each piece of furniture"),
            DataContent(uri=image_uri, media_type="image/png")
        ]
)

In [ ]:
result=''
async for event in workflow.run_stream(message):
    if isinstance(event, DatabaseEvent):
        print(f"{event}")
    if isinstance(event, WorkflowEvent):
        result += str(event.data)

In [ ]:
result.replace("None", "")


---

**Descargo de responsabilidad**:  
Este documento ha sido traducido utilizando el servicio de traducci?n autom?tica [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por garantizar la precisi?n, tenga en cuenta que las traducciones automatizadas pueden contener errores o imprecisiones. El documento original en su idioma nativo debe considerarse como la fuente autorizada. Para informaci?n cr?tica, se recomienda una traducci?n profesional realizada por humanos. No nos hacemos responsables de malentendidos o interpretaciones err?neas que puedan surgir del uso de esta traducci?n.
